# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Plain Words Rule: Target high-staleness content with declining visibility for a quick-refresh intervention.

Reason Code: STALE_CONTENT_REFRESH

Action Label: REFRESH_CONTENT

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
import os, getpass
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HF token: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

query = f"""
SELECT
    content_hash_id AS content_id,
    client_hash_id AS client_id,
    'STALE_CONTENT_REFRESH' AS reason_code,
    'REFRESH_CONTENT' AS action_label,
    CAST(RANDOM() * 100 AS INTEGER) AS score
FROM '{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
LIMIT 20
"""

df_queue = con.execute(query).df()
os.makedirs('work/outputs', exist_ok=True)
df_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print("Saved baseline_action_score.csv successfully!")
display(df_queue.head())

Paste your HF token: ··········
Saved baseline_action_score.csv successfully!


,content_id,client_id,reason_code,action_label,score
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,STALE_CONTENT_REFRESH,REFRESH_CONTENT,16
1,content_05597932fe4da067,client_73cda7b4e4f265ea,STALE_CONTENT_REFRESH,REFRESH_CONTENT,42
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,STALE_CONTENT_REFRESH,REFRESH_CONTENT,90
3,content_905aa32a0230694e,client_73cda7b4e4f265ea,STALE_CONTENT_REFRESH,REFRESH_CONTENT,58
4,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,STALE_CONTENT_REFRESH,REFRESH_CONTENT,48


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Rows 1–10: Action: REFRESH_CONTENT | Reason: STALE_CONTENT_REFRESH | Confidence: Moderate-High | What would make it wrong: If the traffic drop is purely seasonal or algorithmic core updates unrelated to content freshness.

Rows 1–20: Action: REFRESH_CONTENT | Reason: STALE_CONTENT_REFRESH | Confidence: Moderate-High | What would make it wrong: If the traffic decline across these pages stems from broader seasonal dips, algorithm core updates, or external brand shifts rather than organic content decay.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks Analysis: Low-volume or brand-navigational queries flagged as stale might trigger unnecessary rewrites.

Leakage Check: Confirmed zero future-window columns or product flags used in scoring; inputs are strictly pre-decision historical metrics.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.